# 02 Anomaly Detector

Refactors the pilot notebook historical-feature scoring, strong anomaly classification, ranking, and output logic. It uses existing processed pilot features when present, preserving the original pilot notebook unchanged.


## Setup


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "falnama_config.yaml").exists() or (candidate / "polymarket_geopolitics_anomaly_detection_pilot.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate Falnama project root. Run from inside the Falnama folder.")

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config" / "falnama_config.yaml"
RUN_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def load_config() -> dict[str, Any]:
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

CONFIG = load_config()
REPOSITORIES = CONFIG.get("repositories", {})
for repo_rel in REPOSITORIES.values():
    (PROJECT_ROOT / repo_rel).mkdir(parents=True, exist_ok=True)

RUN_LOG_DIR = PROJECT_ROOT / REPOSITORIES.get("run_logs", "repositories/run_logs")
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)


def repo_path(key: str, default: str) -> Path:
    path = PROJECT_ROOT / REPOSITORIES.get(key, default)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_run_log(notebook_name: str, records: list[dict[str, Any]]) -> Path:
    path = RUN_LOG_DIR / f"{notebook_name}_{RUN_TIME_UTC.replace(':', '').replace('-', '')}.jsonl"
    with path.open("x", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps({"run_time_utc": RUN_TIME_UTC, **record}, default=str) + "\n")
    return path


def load_csv_nonempty(path: Path) -> pd.DataFrame | None:
    if not path.exists() or path.stat().st_size <= 1:
        return None
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return None
    return df if not df.empty else None


def normalize_timestamp(value: Any) -> pd.Timestamp:
    if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
        return pd.NaT
    if isinstance(value, pd.Timestamp):
        return value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")
    if isinstance(value, (int, float, np.integer, np.floating)):
        unit = "ms" if float(value) > 10_000_000_000 else "s"
        return pd.to_datetime(value, unit=unit, utc=True, errors="coerce")
    return pd.to_datetime(value, utc=True, errors="coerce")


def parse_jsonish(value: Any, default: Any = None) -> Any:
    if default is None:
        default = []
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return default
        try:
            return json.loads(text)
        except Exception:
            return value
    return value


def first_present(obj: dict[str, Any] | pd.Series, keys: list[str], default: Any = None) -> Any:
    for key in keys:
        if key in obj and obj[key] not in (None, "") and not (isinstance(obj[key], float) and np.isnan(obj[key])):
            return obj[key]
    return default


def safe_slug(value: Any, fallback: str = "unknown") -> str:
    text = str(value if value not in (None, "") else fallback).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text[:100] or fallback


## Anomaly Functions
This stage keeps the pilot scoring concepts: robust movement features, deterministic 0-100 component scores, strong/medium/weak classification, cooldown deduplication, and ranked outputs.


In [2]:
ANOMALY_CFG = CONFIG.get("anomaly_detector", {})
LEGACY = CONFIG.get("legacy_inputs", {})
RELEVANT_DIR = repo_path("relevant_markets", "repositories/relevant_markets")
REJECTED_DIR = repo_path("rejected_signals", "repositories/rejected_signals")
LOGS: list[dict[str, Any]] = []


def robust_z(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce")
    med = values.median(skipna=True)
    mad = (values - med).abs().median(skipna=True)
    if pd.isna(mad) or mad == 0:
        std = values.std(skipna=True)
        if pd.isna(std) or std == 0:
            return pd.Series(np.zeros(len(values)), index=values.index, dtype=float)
        return (values - med) / std
    return 0.6745 * (values - med) / mad


def classify_score(score: float) -> str:
    if pd.isna(score):
        return "none"
    if score >= float(ANOMALY_CFG.get("strong_score_threshold", 85)):
        return "strong"
    if score >= float(ANOMALY_CFG.get("medium_score_threshold", 70)):
        return "medium"
    if score >= float(ANOMALY_CFG.get("weak_score_threshold", 50)):
        return "weak"
    return "none"


def rescale_0_100(values: pd.Series) -> pd.Series:
    values = pd.to_numeric(values, errors="coerce")
    finite = values[np.isfinite(values)]
    if finite.empty:
        return pd.Series(np.zeros(len(values)), index=values.index, dtype=float)
    lo, hi = finite.min(), finite.max()
    if hi == lo:
        return pd.Series(np.full(len(values), 50.0), index=values.index, dtype=float)
    return ((values - lo) / (hi - lo) * 100).clip(0, 100).fillna(0)


def compute_scores_from_features(features: pd.DataFrame) -> pd.DataFrame:
    df = features.copy()
    for col in ["timestamp_utc", "close_time"]:
        if col in df.columns:
            df[col] = df[col].apply(normalize_timestamp)
    score_inputs = {
        "speed_score": rescale_0_100(df.get("speed_1h", pd.Series(index=df.index, dtype=float))),
        "magnitude_score": rescale_0_100(df.get("max_abs_delta_window", pd.Series(index=df.index, dtype=float))),
        "persistence_score": (pd.to_numeric(df.get("persistence_1h", pd.Series(index=df.index, dtype=float)), errors="coerce").clip(-2, 3).fillna(0) + 2) / 5 * 100,
        "liquidity_adjusted_impact_score": rescale_0_100(df.get("liquidity_adjusted_impact_proxy", pd.Series(index=df.index, dtype=float))),
        "time_to_close_score": pd.to_numeric(df.get("time_to_close_score", pd.Series(index=df.index, dtype=float)), errors="coerce").clip(0, 100).fillna(30),
        "concentration_score": pd.to_numeric(df.get("concentration_score", pd.Series(index=df.index, dtype=float)), errors="coerce").clip(0, 100),
    }
    df = df.assign(**score_inputs)
    concentration = df["concentration_score"].fillna(0)
    concentration_weight = np.where(df.get("concentration_data_available", False).astype(bool), 0.15, 0.0)
    base_weight = 1 - concentration_weight
    df["overall_anomaly_score"] = (
        base_weight * (
            0.25 * df["speed_score"] +
            0.30 * df["magnitude_score"] +
            0.20 * df["persistence_score"] +
            0.15 * df["liquidity_adjusted_impact_score"] +
            0.10 * df["time_to_close_score"]
        ) + concentration_weight * concentration
    ).clip(0, 100)
    df["anomaly_class"] = df["overall_anomaly_score"].map(classify_score)
    df["direction"] = np.where(pd.to_numeric(df.get("delta_1h", 0), errors="coerce") >= 0, "up", "down")
    df["anomaly_timestamp"] = df.get("timestamp_utc")
    df["market_url"] = df.get("url", df.get("market_url", None))
    df["notes"] = np.where(df.get("concentration_data_available", False).astype(bool), "deterministic score with public concentration", "wallet concentration unavailable; deterministic price/liquidity/timing score only")
    return df


def deduplicate_anomalies(candidates: pd.DataFrame) -> pd.DataFrame:
    if candidates.empty:
        return candidates.copy()
    cooldown = pd.Timedelta(hours=float(ANOMALY_CFG.get("anomaly_cooldown_hours", 6)))
    kept = []
    for token_id, group in candidates.sort_values(["token_id", "anomaly_timestamp"]).groupby("token_id", sort=False):
        selected_times: list[pd.Timestamp] = []
        for _, row in group.sort_values("overall_anomaly_score", ascending=False).iterrows():
            ts = normalize_timestamp(row.get("anomaly_timestamp"))
            if pd.isna(ts) or all(abs(ts - prev) > cooldown for prev in selected_times):
                kept.append(row)
                if not pd.isna(ts):
                    selected_times.append(ts)
    return pd.DataFrame(kept).sort_values("overall_anomaly_score", ascending=False).reset_index(drop=True) if kept else pd.DataFrame()


def final_ranked_columns() -> list[str]:
    return [
        "rank", "overall_anomaly_score", "anomaly_class", "market_id", "event_id", "question",
        "outcome_name", "token_id", "condition_id", "anomaly_timestamp", "direction", "price_before", "price_after",
        "delta_15m", "delta_1h", "delta_6h", "delta_24h", "robust_z_1h", "robust_z_6h",
        "speed_score", "magnitude_score", "persistence_score", "liquidity_adjusted_impact_score",
        "time_to_close_score", "concentration_data_available", "concentration_score", "wallet_gini",
        "hhi_wallet_volume", "top_1_wallet_share", "top_3_wallet_share", "top_5_wallet_share",
        "concentration_wallet_count", "concentration_trade_count", "concentration_total_notional_proxy",
        "selected_concentration_window_minutes", "selected_concentration_scope", "selected_concentration_missingness_reason",
        "largest_wallet_notional_proxy", "buy_notional_share", "sell_notional_share", "trade_scope",
        "concentration_window_start_utc", "concentration_window_end_utc", "volume", "liquidity", "hours_to_close",
        "time_to_close_bucket", "market_url", "notes"
    ]


def final_ranked_table(deduped: pd.DataFrame) -> pd.DataFrame:
    cols = final_ranked_columns()
    if deduped.empty:
        return pd.DataFrame(columns=cols)
    retained = deduped[deduped["anomaly_class"].isin(["weak", "medium", "strong"])].copy()
    retained = retained.sort_values("overall_anomaly_score", ascending=False).head(int(ANOMALY_CFG.get("top_n_anomalies", 100))).reset_index(drop=True)
    retained.insert(0, "rank", np.arange(1, len(retained) + 1))
    for col in cols:
        if col not in retained.columns:
            retained[col] = np.nan
    return retained[cols]


def load_or_compute_anomalies() -> tuple[pd.DataFrame, str]:
    """Prefer the pilot's canonical ranked output when it exists.

    The pilot already produced ranked/strong anomaly files from the full retrieval,
    price-history, movement, concentration, classification, and ranking workflow.
    This notebook keeps the refactored scoring functions above for fresh/fallback
    runs, but it does not discard known-good pilot rankings when they are present.
    """
    ranked = load_csv_nonempty(PROJECT_ROOT / LEGACY.get("legacy_ranked_anomalies", ""))
    if ranked is not None:
        return ranked, "copied_from_legacy_ranked_anomalies"
    features = load_csv_nonempty(PROJECT_ROOT / LEGACY.get("legacy_anomaly_features", ""))
    if features is not None:
        LOGS.append({"event": "features_loaded", "rows": len(features)})
        scored = compute_scores_from_features(features)
        min_obs = int(ANOMALY_CFG.get("min_price_observations", 20))
        candidates = scored[
            (pd.to_numeric(scored.get("price_observation_count", 0), errors="coerce") >= min_obs) &
            (scored["anomaly_class"].isin(["weak", "medium", "strong"]))
        ].copy()
        return final_ranked_table(deduplicate_anomalies(candidates)), "computed_from_legacy_features"
    if ANOMALY_CFG.get("use_mock_if_no_inputs", True):
        mock = pd.DataFrame([{
            "rank": 1,
            "overall_anomaly_score": 92.0,
            "anomaly_class": "strong",
            "market_id": "mock-market-001",
            "event_id": "mock-event-001",
            "question": "Mock geopolitical escalation market for smoke testing",
            "outcome_name": "Yes",
            "token_id": "mock-token-yes",
            "condition_id": "mock-condition-001",
            "anomaly_timestamp": RUN_TIME_UTC,
            "direction": "up",
            "price_before": 0.35,
            "price_after": 0.55,
            "delta_1h": 0.20,
            "robust_z_1h": 8.0,
            "market_url": None,
            "notes": "mock anomaly because no real inputs were found",
        }])
        return mock, "mock"
    return pd.DataFrame(columns=final_ranked_columns()), "empty"



## Run Anomaly Detector
Writes ranked and strong anomaly CSVs to `repositories/relevant_markets/` for downstream stages.


In [3]:
ranked_anomalies_df, source_mode = load_or_compute_anomalies()
strong_anomalies_df = ranked_anomalies_df[ranked_anomalies_df.get("anomaly_class", "").astype(str).str.lower() == "strong"].copy() if not ranked_anomalies_df.empty else pd.DataFrame()

stamp = RUN_TIME_UTC.replace(":", "").replace("-", "")
ranked_path = RELEVANT_DIR / f"ranked_anomalies_{stamp}.csv"
strong_path = RELEVANT_DIR / f"strong_anomalies_{stamp}.csv"
ranked_anomalies_df.to_csv(ranked_path, index=False)
strong_anomalies_df.to_csv(strong_path, index=False)
log_path = write_run_log("02_anomaly_detector", LOGS + [{"event": "anomaly_outputs", "source_mode": source_mode, "ranked_rows": len(ranked_anomalies_df), "strong_rows": len(strong_anomalies_df), "ranked_path": str(ranked_path), "strong_path": str(strong_path)}])

print(f"Anomaly source mode: {source_mode}")
print(f"Ranked anomalies: {len(ranked_anomalies_df):,}")
print(f"Strong anomalies: {len(strong_anomalies_df):,}")
print(ranked_path)
print(strong_path)
print(log_path)
display(ranked_anomalies_df.head(20))


Anomaly source mode: copied_from_legacy_ranked_anomalies
Ranked anomalies: 100
Strong anomalies: 24
/Users/R2-D2/Documents/Codex/Falnama/repositories/relevant_markets/ranked_anomalies_20260609T003752Z.csv
/Users/R2-D2/Documents/Codex/Falnama/repositories/relevant_markets/strong_anomalies_20260609T003752Z.csv
/Users/R2-D2/Documents/Codex/Falnama/repositories/run_logs/02_anomaly_detector_20260609T003752Z.jsonl


,rank,overall_anomaly_score,anomaly_class,market_id,event_id,question,outcome_name,token_id,condition_id,anomaly_timestamp,direction,price_before,price_after,delta_15m,delta_1h,delta_6h,delta_24h,robust_z_1h,robust_z_6h,speed_score,magnitude_score,persistence_score,liquidity_adjusted_impact_score,time_to_close_score,concentration_data_available,concentration_score,wallet_gini,hhi_wallet_volume,top_1_wallet_share,top_3_wallet_share,top_5_wallet_share,concentration_wallet_count,concentration_trade_count,concentration_total_notional_proxy,selected_concentration_window_minutes,selected_concentration_scope,selected_concentration_missingness_reason,largest_wallet_notional_proxy,buy_notional_share,sell_notional_share,trade_scope,concentration_window_start_utc,concentration_window_end_utc,volume,liquidity,hours_to_close,time_to_close_bucket,market_url,notes
0,1,95.204947,strong,923041,103730,Will People’s Party (PPLE) win the most seats in the 2026 Thai legislative election?,No,114665146755612568105376188460288931180214626216373863827528325594298433114727,0x6a79204ace08d896a72cac89c3c2b4c64b90218838c213e9377860596dc77ac5,2026-02-08 13:00:40+00:00,up,0.3600,0.9050,0.3100,0.5450,0.7350,0.6900,72.846000,32.151167,100.0,100.000,90.289908,100.0,76.324697,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,unavailable,no_trades_in_time_window,NaN,NaN,NaN,unavailable,2026-02-08 12:45:40+00:00,2026-02-08 13:00:40+00:00,6.981656e+06,NaN,19.452500,6-24h,https://polymarket.com/event/thai-legislative-election-winner/will-peoples-party-pple-win-the-most-seats-in-the-2026-thai-legislative-election,wallet concentration unavailable; deterministic price/liquidity/timing score only
1,2,95.204888,strong,923041,103730,Will People’s Party (PPLE) win the most seats in the 2026 Thai legislative election?,Yes,50178306595800060471430951874976657618055148106679384594654922816495456398170,0x6a79204ace08d896a72cac89c3c2b4c64b90218838c213e9377860596dc77ac5,2026-02-08 13:00:38+00:00,down,0.6400,0.0950,-0.3100,-0.5450,-0.7350,-0.6900,72.846000,32.151167,100.0,100.000,90.289908,100.0,76.324109,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,unavailable,no_trades_in_time_window,NaN,NaN,NaN,unavailable,2026-02-08 12:45:38+00:00,2026-02-08 13:00:38+00:00,6.981656e+06,NaN,19.453056,6-24h,https://polymarket.com/event/thai-legislative-election-winner/will-peoples-party-pple-win-the-most-seats-in-the-2026-thai-legislative-election,wallet concentration unavailable; deterministic price/liquidity/timing score only
2,3,95.190335,strong,923042,103730,Will Bhumjaithai Party (BJT) win the most seats in the 2026 Thai legislative election?,Yes,94912568711096777820489984696016798235367443551768917424847805010888215349861,0x91fd3b7cf10e925f169ff49dd657baf9794025b241965fecd0c1b9cbd9c1e115,2026-02-08 12:00:55+00:00,up,0.3600,0.6250,0.2650,0.2650,0.4500,0.4500,35.074000,29.340750,100.0,93.500,100.000000,100.0,74.653348,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,unavailable,no_trades_in_time_window,NaN,NaN,NaN,unavailable,2026-02-08 11:45:55+00:00,2026-02-08 12:00:55+00:00,3.057801e+06,NaN,21.046667,6-24h,https://polymarket.com/event/thai-legislative-election-winner/will-bhumjaithai-party-bjt-win-the-most-seats-in-the-2026-thai-legislative-election,wallet concentration unavailable; deterministic price/liquidity/timing score only
3,4,95.190104,strong,923042,103730,Will Bhumjaithai Party (BJT) win the most seats in the 2026 Thai legislative election?,No,22926980609547969135527904525228886157524757300125230214542888682865237192341,0x91fd3b7cf10e925f169ff49dd657baf9794025b241965fecd0c1b9cbd9c1e115,2026-02-08 12:00:47+00:00,down,0.6400,0.3750,-0.2650,-0.2650,-0.4500,-0.4500,35.074000,26.673409,100.0,93.500,100.000000,100.0,74.651044,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,unavailable,no_trades_in_time_window,NaN,NaN,NaN,unavailable,2026-02-08 11:45:47+00:00,2026-02-08 12:00:47+00:00,3.057801e+06,NaN,21.048889,6-24h,https://polymarket.com/event/thai-legislative-election-winner/will-bhumjaithai-party-